# Elo Exploration for EPL-like ProjectThis notebook builds a light Elo scaffold, loads data from elo_project_data.md or CSVs in data/epl/, and includes a simple data-preview, plotting, and usage notes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Data loading: try elo_project_data.md, else CSVs in data/epl/
try:
    with open('elo_project_data.md','r') as f:
        text = f.read()
    # keep it simple: show first lines as preview
    data_preview = text[:1000]
    print('Loaded elo_project_data.md preview:', data_preview[:200])
    data = None
except FileNotFoundError:
    # attempt CSVs in data/epl/
    import os
    csvs = []
    for root,dirs,files in os.walk('data/epl'):
        for f in files:
            if f.endswith('.csv'):
                csvs.append(os.path.join(root,f))
    if csvs:
        data = pd.concat([pd.read_csv(p) for p in csvs], ignore_index=True)
    else:
        data = pd.DataFrame()
    print('Data shape:', data.shape)

# Hard-coded dummy until real data is loaded?

In [ ]:
# Simple Elo scaffold: start 1500, per-match update with K, time-decay placeholder
import math

K = 40
INITIAL_ELO = 1500
elo = {}  # player:str -> Elo value

# suppose data has columns: winner, loser, date (YYYY-MM-DD) - this is a placeholder
if data is not None and not data.empty:
    # ensure required columns exist or create mock if missing
    for c in ['winner','loser','date']:
        if c not in data.columns:
            data[c] = None

# Build players list from data if possible, else build a tiny sample
players = set()
if data is not None and not data.empty:
    for p in pd.concat([data['winner'], data['loser']]).astype(str).tolist():
        if p:
            players.add(p)
# Initialize Elo for observed players
for p in players:
    elo[p] = INITIAL_ELO
# Simple per-match update function

def update_elo(winner, loser, k=K):
    if winner not in elo:
        elo[winner] = INITIAL_ELO
    if loser not in elo:
        elo[loser] = INITIAL_ELO
    # expected scores
    def exp(a,b):
        return 1/(1+10**((elo[b]-elo[a])/400))
    Ew = exp(elo[winner], elo[loser])
    El = exp(elo[loser], elo[winner])
    elo[winner] = elo[winner] + k*(1-Ew)
    elo[loser] = elo[loser] + k*(0-El)
    return elo[winner], elo[loser]]


In [ ]:
# Data preview: show first few rows or markdown summary
print('Players:', sorted(list(elo.keys())))
print('Top Elo snapshot:', sorted([(p, elo[p]) for p in elo], key=lambda x:-x[1])[:5])

# If data has matches, apply updates (safe guard if no data)
if data is not None and not data.empty:
    matches = data.dropna(subset=['winner','loser','date'])[[ 'winner','loser','date']].astype(str)
    # Sort by date if available
    try:
        matches['date'] = pd.to_datetime(matches['date'], errors='coerce')
        matches = matches.sort_values('date')
    except Exception:
        pass
    for _, row in matches.iterrows():
        w = str(row['winner']); l = str(row['loser'])
        update_elo(w,l)

# Plot preview
names = sorted(elo.keys())
values = [elo[n] for n in names]
plt.figure(figsize=(8,4))
plt.bar(names, values, color='C0')
plt.xticks(rotation=90)
plt.ylabel('Elo')
plt.title('Elo by Player (sample)')
plt.tight_layout()
plt.show()